# Forensic Guard — Validation Notebook

**Beneish M-Score:** Reduced 6-factor model (GMI and SGAI unavailable from Tickertape).  
**Altman Z-Score:** Full Z'' (emerging market variant) with PBT+interest as EBIT proxy.

**Data gap workarounds:**
- COGS (for GMI): not available → component dropped, thresholds adjusted +0.50
- SGA (for SGAI): not available → component dropped
- operating_profit: 100% NULL → use PBT + interest as EBIT proxy for Z-score X3

**v1 covered 399 stocks (yfinance). v2 covers ~2,000+ (Tickertape fundamentals).**

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
from signals.forensic import _load_data, _compute_scores
from db import read_sql

stocks, financial_sids, qi, bs, cf = _load_data()
v2 = _compute_scores(stocks, financial_sids, qi, bs, cf)

print(f"v2: {len(v2)} stocks")
print(f"  M-Score (6-factor): {v2['m_score'].notna().sum()} (v1 had 399)")
print(f"  Z-Score (Z''): {v2['z_score'].notna().sum()}")
print(f"\nM-Score: mean={v2['m_score'].dropna().mean():.2f}, median={v2['m_score'].dropna().median():.2f}")
print(f"Z-Score: mean={v2['z_score'].dropna().mean():.2f}, median={v2['z_score'].dropna().median():.2f}")
print(f"\nM flags: {v2['m_score_flag'].value_counts().to_dict()}")
print(f"Z flags: {v2['z_score_flag'].value_counts().to_dict()}")
print(f"Penalized: {(v2['penalty'] < 0).sum()}")

# Per-tier breakdown
tiers = read_sql("SELECT sid, cap_tier FROM stocks")
v2m = v2.merge(tiers, on="sid")
for tier in ["LARGE", "MID", "SMALL"]:
    t = v2m[v2m["cap_tier"] == tier]
    n = t["m_score"].notna().sum()
    flagged = t["m_score_flag"].isin(["LIKELY_MANIPULATOR", "POSSIBLE_MANIPULATOR"]).sum()
    print(f"  {tier}: {n} scored, {flagged} flagged ({flagged/n*100:.0f}%)" if n > 0 else f"  {tier}: 0")

## Compare with v1 (399 overlapping stocks)

In [ ]:
v1 = pd.read_csv(os.path.expanduser("~/alpha-signal/data/forensic/forensic_scores.csv"))
ticker_map = read_sql("SELECT sid, ticker FROM stocks").set_index("ticker")["sid"].to_dict()
v1["sid"] = v1["symbol"].map(ticker_map)
v1 = v1.dropna(subset=["sid"])
merged = v2.merge(v1[["sid", "m_score", "z_score", "m_flag", "z_flag"]],
                  on="sid", how="inner", suffixes=("_v2", "_v1"))

for label, v2c, v1c in [("Z-Score", "z_score_v2", "z_score_v1"), ("M-Score", "m_score_v2", "m_score_v1")]:
    both = merged[[v2c, v1c]].dropna()
    corr = both.iloc[:, 0].corr(both.iloc[:, 1])
    diff = (both.iloc[:, 0] - both.iloc[:, 1]).abs().mean()
    print(f"{label}: {len(both)} stocks, corr={corr:.4f}, mean|diff|={diff:.2f}")

## Save to DB (only if validation passed)

In [ ]:
# from signals.forensic import compute
# compute(dry_run=False)